In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop('customerID',axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Convert TotalCharges to numeric

In [ ]:
df.isnull().sum()

In [ ]:
# Fill missing TotalCharges with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
columns = df.columns

In [ ]:
for col in columns:
  print(f'{col}: {df[col].unique()}')

In [ ]:
def get_services(row):
    services = []

    if row['PhoneService'] == 'Yes':
        services.append(1)

    if row['MultipleLines'] == 'Yes':
        services.append(1)

    if row['InternetService'] != 'No':
        services.append(1)  # 'DSL' or 'Fiber optic'

    # These depend on internet service
    internet_features = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                         'TechSupport', 'StreamingTV', 'StreamingMovies']

    for feature in internet_features:
        if row[feature] == 'Yes':
            services.append(1)

    return len(services)+1

# Apply to DataFrame
df['services'] = df.apply(get_services, axis=1)

In [ ]:
df.head()

In [ ]:
df.drop(['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
# Encode categorical features
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].apply(LabelEncoder().fit_transform)

In [ ]:
df.head()

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = AdaBoostClassifier(n_estimators=50, learning_rate=0.2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.2, 0.3]
}
search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy')
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best score:", search.best_score_)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'bootstrap': [True, False],
    'max_samples': [0.5, 0.75, 1.0]
}
search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy')
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best score:", search.best_score_)

In [ ]:
import plotly.express as px
fig = px.histogram(df, x="services", color="Churn", barmode='stack')
fig.show()